# Day 3 — Contract search, the tiny version

The whole idea in four moves:

1. **Chop** the contracts into small pieces of text.
2. **Embed** — turn each piece into a list of numbers that captures its *meaning*.
3. **Store** those number-lists in a FAISS index so we can search them fast.
4. **Search** — turn your question into numbers the same way, and find the closest pieces.

That's the entire thing. The big reference notebook is just this with extra polish on top.

Run this once to install the three tools we need, then restart the kernel if asked.

In [ ]:
%pip install -q faiss-cpu sentence-transformers "numpy<2.0"

## The whole engine

Read it top to bottom — each block is one of the four moves above.

In [ ]:
import glob, faiss
from sentence_transformers import SentenceTransformer

# 1. CHOP every contract into ~200-word pieces
pieces = []
for path in glob.glob("contracts/*.txt"):
    words = open(path, encoding="utf-8").read().split()
    for i in range(0, len(words), 200):
        pieces.append(" ".join(words[i:i + 200]))

# 2. EMBED each piece into a list of numbers (normalize so we can compare by meaning)
model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(pieces, normalize_embeddings=True)

# 3. STORE the vectors in a FAISS index
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print(f"Ready: {len(pieces)} pieces of contract text, searchable.")

## Ask it a question

`ask()` does move 4: it embeds your question and prints the closest pieces. The number in front is the similarity score (higher = closer in meaning). Change the question and run it again.

In [ ]:
def ask(question, k=3):
    q = model.encode([question], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    for score, i in zip(scores[0], ids[0]):
        print(f"({score:.2f}) {pieces[i][:150].strip()}...\n")


ask("can I hire their employees after the contract ends?")

Notice the magic: you asked about *hiring employees* and it can surface clauses that say *"solicit, induce, or engage"* — words your question never used. That's what embeddings buy you over plain keyword search.

Try a few of your own:
- `ask("who owns the intellectual property?")`
- `ask("how is confidential information protected?")`

When this feels comfortable, open `Day_03_contract_clause_search_REFERENCE.ipynb` — it's the same four moves, just with cleaner chunking, metadata, and two framework versions layered on.